In [ ]:
import torch
from sae_lens import SAE
from transformer_lens import HookedTransformer
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory

PROMPT = "The capital of France is Paris, which is known for"
RELEASE = "gpt2-small-res-jb"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

release_saes = get_pretrained_saes_directory()[RELEASE].saes_map
model = HookedTransformer.from_pretrained("gpt2-small", device=DEVICE)

sae_objects = {}
for sid in release_saes.keys():
    sae, cfg, sparsity = SAE.from_pretrained(release=RELEASE, sae_id=sid, device=DEVICE)
    sae_objects[sid] = sae

tokens = model.to_tokens(PROMPT)
logits, cache = model.run_with_cache(tokens)

for sid, hook_point in release_saes.items():
    print(f"\n=== {hook_point} ===")
    sae = sae_objects[sid]
    acts = cache[hook_point] 
    feature_acts = sae.encode(acts)
    
    for t in range(feature_acts.shape[1]):
        token_acts = feature_acts[0, t]
        active_indices = (token_acts > 0).nonzero(as_tuple=True)[0]
        if len(active_indices) > 0:
            active_vals = token_acts[active_indices]
            sorted_sort_idx = torch.argsort(active_vals, descending=True)
            sorted_indices = active_indices[sorted_sort_idx].tolist()
            sorted_vals = active_vals[sorted_sort_idx].tolist()
            sum_vals = sum(sorted_vals)
            rel_vals = [v / sum_vals for v in sorted_vals] if sum_vals > 0 else []
        else:
            sorted_indices = []
            sorted_vals = []
            rel_vals = []
            
        print(f"  pos {t}: {sorted_indices}")
        print(f"  pos {t} magnitudes: {sorted_vals}")
        print(f"  pos {t} relative: {rel_vals}")